In [ ]:
# Install PySpark in Google Colab
!pip -q install pyspark

In [ ]:
# Import libraries and create the Data Lake directories
import os
import shutil
import glob
from pyspark.sql import SparkSession, functions as F

BASE_DIR = "/content/data_lake"
BRONZE_DIR = f"{BASE_DIR}/bronze"
SILVER_DIR = f"{BASE_DIR}/silver"
SILVER_SPARK_DIR = f"{SILVER_DIR}/preprocessed_sales"
BRONZE_FILE = f"{BRONZE_DIR}/sales.csv"
SILVER_FILE = f"{SILVER_DIR}/sales_silver.csv"

os.makedirs(BRONZE_DIR, exist_ok=True)
os.makedirs(SILVER_DIR, exist_ok=True)

# The class repository contains the authoritative dataset.
DATASET_URL = "https://raw.githubusercontent.com/necromancersanju-netizen/data-mining-data-lake-assignment/main/dataset/sales.csv"

# Download the raw dataset directly into Bronze.
# Do not edit this file after download.
!wget -q -O /content/data_lake/bronze/sales.csv $DATASET_URL

print("Bronze file:", BRONZE_FILE)
print("Bronze file exists:", os.path.exists(BRONZE_FILE))
print("Bronze file size (bytes):", os.path.getsize(BRONZE_FILE))

In [ ]:
# Create a SparkSession
spark = (
    SparkSession.builder
    .appName("DataMiningSparkDataLakeAssignment")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

In [ ]:
# Read the Bronze dataset with Spark
raw_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("mode", "PERMISSIVE")
    .csv(BRONZE_FILE)
)

print("Schema:")
raw_df.printSchema()

print("First 5 rows:")
raw_df.show(5, truncate=False)

raw_count = raw_df.count()
print("Total records:", raw_count)

In [ ]:
# 2.1 Exact duplicate rows
unique_count = raw_df.dropDuplicates().count()
exact_duplicate_rows = raw_count - unique_count

print("Exact duplicate rows to remove:", exact_duplicate_rows)

In [ ]:
# 2.2 Null values by column
null_counts = raw_df.select(*[
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in raw_df.columns
])

print("Null values by column:")
null_counts.show(truncate=False)

In [ ]:
# Helper for checking obvious capitalization/spacing inconsistencies
CANONICAL_VALUES = {
    "category": [
        "Clothing", "Sports", "Grocery", "Books", "Furniture", "Beauty", "Electronics", "Home Appliances"
    ],
    "payment_method": [
        "Credit Card", "Debit Card", "UPI", "Net Banking", "Cash on Delivery"
    ],
    "city": [
        "Mumbai", "Chennai", "Pune", "Lucknow", "Vijayawada", "Hyderabad", "Delhi", "Jaipur", "Kolkata", "Coimbatore", "Bengaluru"
    ],
    "state": [
        "Maharashtra", "Tamil Nadu", "Uttar Pradesh", "Andhra Pradesh", "Telangana", "Delhi", "Rajasthan", "West Bengal", "Karnataka"
    ],
    "order_status": [
        "Delivered", "Shipped", "Cancelled", "Processing", "Returned"
    ]
}

def canonical_text_expr(column, values):
    lowered = F.lower(F.trim(F.col(column).cast("string")))
    expr = F.when(F.col(column).isNull(), F.lit(None).cast("string"))
    for value in values:
        expr = expr.when(lowered == value.lower(), F.lit(value))
    return expr.otherwise(F.initcap(F.trim(F.col(column).cast("string"))))

text_check_exprs = []
for col_name, values in CANONICAL_VALUES.items():
    standardized = canonical_text_expr(col_name, values)
    raw_text = F.col(col_name).cast("string")
    text_check_exprs.append(
        F.sum(
            F.when(
                raw_text.isNotNull() & (F.trim(raw_text) != standardized),
                1
            ).otherwise(0)
        ).alias(col_name)
    )

text_inconsistency_counts = raw_df.select(*text_check_exprs)
print("Obvious text-format inconsistencies (spacing/capitalization):")
text_inconsistency_counts.show(truncate=False)

In [ ]:
# 2.4 Parse numeric fields for validation
numeric_preview = (
    raw_df
    .withColumn("quantity_num", F.col("quantity").cast("double"))
    .withColumn("unit_price_num", F.col("unit_price").cast("double"))
    .withColumn("discount_num", F.regexp_replace(F.col("discount_percent").cast("string"), "%", "").cast("double"))
)

qty_raw = F.trim(F.col("quantity").cast("string"))
price_raw = F.trim(F.col("unit_price").cast("string"))
disc_raw = F.trim(F.regexp_replace(F.col("discount_percent").cast("string"), "%", ""))

invalid_quantity = qty_raw.isNotNull() & (qty_raw != "") & (
    F.col("quantity_num").isNull() | (F.col("quantity_num") <= 0)
)
invalid_unit_price = price_raw.isNotNull() & (price_raw != "") & (
    F.col("unit_price_num").isNull() | (F.col("unit_price_num") <= 0)
)
invalid_discount = disc_raw.isNotNull() & (disc_raw != "") & (
    F.col("discount_num").isNull() | (F.col("discount_num") < 0) | (F.col("discount_num") > 100)
)

numeric_issue_counts = numeric_preview.select(
    F.sum(F.when(invalid_quantity, 1).otherwise(0)).alias("invalid_quantity"),
    F.sum(F.when(invalid_unit_price, 1).otherwise(0)).alias("invalid_unit_price"),
    F.sum(F.when(invalid_discount, 1).otherwise(0)).alias("invalid_discount")
)

print("Invalid numerical values:")
numeric_issue_counts.show()

In [ ]:
# 2.5 Date-format and invalid-date check
parsed_date_preview = (
    raw_df
    .withColumn(
        "parsed_order_date",
        F.coalesce(
            F.expr("try_to_timestamp(trim(cast(order_date as string)), 'yyyy-MM-dd')"),
            F.expr("try_to_timestamp(trim(cast(order_date as string)), 'yyyy/MM/dd')"),
            F.expr("try_to_timestamp(trim(cast(order_date as string)), 'dd/MM/yyyy')"),
            F.expr("try_to_timestamp(trim(cast(order_date as string)), 'dd-MM-yyyy')"),
            F.expr("try_to_timestamp(trim(cast(order_date as string)), 'MM/dd/yyyy')"),
            F.expr("try_to_timestamp(trim(cast(order_date as string)), 'MM-dd-yyyy')")
        ).cast("date")
    )
)

invalid_date = (
    F.col("order_date").isNotNull()
    & (F.trim(F.col("order_date").cast("string")) != "")
    & F.col("parsed_order_date").isNull()
)

inconsistent_date_format = (
    F.col("order_date").isNotNull()
    & (F.trim(F.col("order_date").cast("string")) != "")
    & F.col("parsed_order_date").isNotNull()
    & (F.trim(F.col("order_date").cast("string")) != F.date_format(F.col("parsed_order_date"), "yyyy-MM-dd"))
)

date_issue_counts = parsed_date_preview.select(
    F.sum(F.when(inconsistent_date_format, 1).otherwise(0)).alias("valid_dates_needing_standardization"),
    F.sum(F.when(invalid_date, 1).otherwise(0)).alias("invalid_dates")
)

print("Date issues:")
date_issue_counts.show()


In [ ]:
# 3.1 Remove exact duplicates first
silver_df = raw_df.dropDuplicates()

# 3.2 Trim general text fields and handle missing text values
TEXT_COLUMNS = [
    "customer_name", "product", "category", "payment_method", "city", "state", "order_status"
]

for c in TEXT_COLUMNS:
    silver_df = silver_df.withColumn(c, F.trim(F.col(c).cast("string")))

# Standardize obvious categorical text values while preserving useful product/name capitalization.
for c, values in CANONICAL_VALUES.items():
    silver_df = silver_df.withColumn(c, canonical_text_expr(c, values))

silver_df = silver_df.fillna({c: "Unknown" for c in TEXT_COLUMNS})

# Empty strings are treated as missing text as well.
for c in TEXT_COLUMNS:
    silver_df = silver_df.withColumn(
        c,
        F.when(F.trim(F.col(c)) == "", F.lit("Unknown")).otherwise(F.col(c))
    )

In [ ]:
# 3.3 Validate numeric fields
silver_df = (
    silver_df
    .withColumn("quantity_num", F.col("quantity").cast("double"))
    .withColumn("unit_price_num", F.col("unit_price").cast("double"))
    .withColumn(
        "discount_num",
        F.regexp_replace(F.col("discount_percent").cast("string"), "%", "").cast("double")
    )
)

# Clearly invalid values become null.
silver_df = silver_df.withColumn(
    "quantity_num",
    F.when(F.col("quantity_num") > 0, F.col("quantity_num"))
)
silver_df = silver_df.withColumn(
    "unit_price_num",
    F.when(F.col("unit_price_num") > 0, F.col("unit_price_num"))
)
silver_df = silver_df.withColumn(
    "discount_num",
    F.when((F.col("discount_num") >= 0) & (F.col("discount_num") <= 100), F.col("discount_num"))
)

# Use the median of valid values to preserve rows with clearly invalid numeric values.
quantity_median = silver_df.approxQuantile("quantity_num", [0.5], 0.0)[0]
unit_price_median = silver_df.approxQuantile("unit_price_num", [0.5], 0.0)[0]
discount_median = silver_df.approxQuantile("discount_num", [0.5], 0.0)[0]

print("Median quantity used for numeric imputation:", quantity_median)
print("Median unit price used for numeric imputation:", unit_price_median)
print("Median discount percentage used for numeric imputation:", discount_median)

silver_df = (
    silver_df
    .withColumn("quantity", F.coalesce(F.col("quantity_num"), F.lit(quantity_median)).cast("int"))
    .withColumn("unit_price", F.coalesce(F.col("unit_price_num"), F.lit(unit_price_median)).cast("double"))
    .withColumn("discount_percent", F.coalesce(F.col("discount_num"), F.lit(discount_median)).cast("double"))
    .drop("quantity_num", "unit_price_num", "discount_num")
)

In [ ]:
# 3.4 Standardize dates
silver_df = (
    silver_df
    .withColumn(
        "parsed_order_date",
        F.coalesce(
            F.expr("try_to_timestamp(trim(cast(order_date as string)), 'yyyy-MM-dd')"),
            F.expr("try_to_timestamp(trim(cast(order_date as string)), 'yyyy/MM/dd')"),
            F.expr("try_to_timestamp(trim(cast(order_date as string)), 'dd/MM/yyyy')"),
            F.expr("try_to_timestamp(trim(cast(order_date as string)), 'dd-MM-yyyy')"),
            F.expr("try_to_timestamp(trim(cast(order_date as string)), 'MM/dd/yyyy')"),
            F.expr("try_to_timestamp(trim(cast(order_date as string)), 'MM-dd-yyyy')")
        ).cast("date")
    )
    .withColumn(
        "order_date",
        F.when(
            F.col("parsed_order_date").isNotNull(),
            F.date_format(F.col("parsed_order_date"), "yyyy-MM-dd")
        ).otherwise(F.lit(None).cast("string"))
    )
    .drop("parsed_order_date")
)


In [ ]:
# 3.5 Final column order and schema cleanup
FINAL_COLUMNS = [
    "order_id", "customer_id", "customer_name", "product", "category",
    "quantity", "unit_price", "discount_percent", "payment_method",
    "city", "state", "order_date", "order_status"
]

silver_df = silver_df.select(*FINAL_COLUMNS)

print("Silver preview:")
silver_df.show(10, truncate=False)
print("Silver schema:")
silver_df.printSchema()

In [ ]:
# Remove old Silver output if the notebook is rerun.
for path in [SILVER_SPARK_DIR, SILVER_FILE]:
    if os.path.isdir(path):
        shutil.rmtree(path)
    elif os.path.isfile(path):
        os.remove(path)

# Spark writes CSV output as a directory. coalesce(1) gives one data part file.
(
    silver_df
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .option("nullValue", "")
    .csv(SILVER_SPARK_DIR)
)

part_files = glob.glob(f"{SILVER_SPARK_DIR}/part-*.csv")
if not part_files:
    raise FileNotFoundError("Spark did not create a CSV part file in the Silver output directory.")

# Copy the single Spark CSV part to the exact filename requested for GitHub submission.
shutil.copy2(part_files[0], SILVER_FILE)

print("Spark Silver directory:", SILVER_SPARK_DIR)
print("Submission Silver file:", SILVER_FILE)

In [ ]:
# Row count before and after preprocessing
silver_count = silver_df.count()
print("Row count before preprocessing:", raw_count)
print("Row count after preprocessing:", silver_count)
print("Rows removed due to exact duplicates:", raw_count - silver_count)

In [ ]:
# Read the Silver file back with Spark and inspect it
silver_readback = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(SILVER_FILE)
)

print("Silver row count after readback:", silver_readback.count())
print("Silver sample:")
silver_readback.show(5, truncate=False)

In [ ]:
# Verify that the main validation rules now hold.
validation_summary = silver_readback.select(
    F.sum(F.when(F.col("quantity") <= 0, 1).otherwise(0)).alias("quantity_le_0"),
    F.sum(F.when(F.col("unit_price") <= 0, 1).otherwise(0)).alias("unit_price_le_0"),
    F.sum(F.when((F.col("discount_percent") < 0) | (F.col("discount_percent") > 100), 1).otherwise(0)).alias("discount_outside_0_100"),
    F.sum(F.when(F.col("order_date").isNull(), 1).otherwise(0)).alias("invalid_or_unavailable_dates")
)

print("Post-preprocessing validation:")
validation_summary.show()

In [ ]:
# display the final runtime structure - easier to verify the Data Lake layout.
for root, dirs, files in os.walk(BASE_DIR):
    level = root.replace(BASE_DIR, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for file_name in sorted(files):
        print(f"{indent}  {file_name}")

In [ ]:
from google.colab import files
files.download(SILVER_FILE)

In [ ]:
# Stop Spark cleanly when all cells have been completed.
spark.stop()